# 01 — Data Preparation and Sentence Splitting

This notebook covers:

1. Load the public-safe source dataset.
2. Filter the intended review period.
3. Reshape review text fields into one NLP text column.
4. Split review text into sentence-level rows.
5. Remove one-word fragments.
6. Export the sentence-level dataset for manual inspection.

> **Project period:** The project description states 2024–2026. The original pasted code used 2012–2023 in the filter, so this notebook uses configurable `START_YEAR = 2024` and `END_YEAR = 2026`.

In [ ]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path("..")
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bank_employee_reviews_raw.csv"
INTERMEDIATE_DIR = PROJECT_ROOT / "data" / "processed"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2024
END_YEAR = 2026

## Load dataset

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)
print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

## Filter the review period

The source dates are day/month/year, so `dayfirst=True` is used.

In [ ]:
df["review_date"] = pd.to_datetime(
    df["review_date"],
    errors="coerce",
    dayfirst=True
)

df_filtered = df[
    df["review_date"].dt.year.between(START_YEAR, END_YEAR)
].copy()

print("Original rows:", len(df))
print("Filtered rows:", len(df_filtered))
print("Date range:",
      df_filtered["review_date"].min(),
      "to",
      df_filtered["review_date"].max())
display(df_filtered.head())

## Convert review text columns into NLP rows

In [ ]:
text_fields = [
    "review_title",
    "review_body_text",
    "pros_text",
    "cons_text"
]

nlp_df = df_filtered.melt(
    id_vars=[c for c in df_filtered.columns if c not in text_fields],
    value_vars=text_fields,
    var_name="text_content",
    value_name="text"
)

nlp_df = nlp_df.dropna(subset=["text"])
nlp_df = nlp_df[nlp_df["text"].astype(str).str.strip().ne("")].copy()

nlp_df["text_content"] = pd.Categorical(
    nlp_df["text_content"],
    categories=text_fields,
    ordered=True
)

nlp_df = (
    nlp_df
    .sort_values(["id", "text_content"])
    .reset_index(drop=True)
)

nlp_df.insert(0, "review_row_id", range(1, len(nlp_df) + 1))

columns_to_keep = [
    "review_row_id",
    "id",
    "company_name",
    "text_content",
    "text"
]
nlp_df = nlp_df[columns_to_keep]

print("NLP text rows created:", len(nlp_df))
display(nlp_df.head())

## Sentence splitting rules

In [ ]:
SPLIT_TOKEN = "<SPLIT_SENT>"
DOT_TOKEN = "<DOT_KEEP>"
DEC_TOKEN = "<DEC_KEEP>"
PAREN_HYPHEN_TOKEN = "<PAREN_HYPHEN_KEEP>"
PROCON_DASH_TOKEN = "<PROCON_DASH_KEEP>"
DAY_RANGE_HYPHEN_TOKEN = "<DAY_RANGE_HYPHEN_KEEP>"
LABEL_COLON_DASH_TOKEN = "<LABEL_COLON_DASH_KEEP>"
CONJ_COMMA_DASH_TOKEN = "<CONJ_COMMA_DASH_KEEP>"

ABBREVIATIONS = [
    "e.g.", "i.e.", "mr.", "mrs.", "ms.", "dr.", "prof.",
    "inc.", "ltd.", "co.", "corp.", "u.s.", "u.k.", "sdn.", "bhd."
]

def clean_space(text):
    return " ".join((text or "").split())

def layer1_prepare_and_split(text):
    t = (text or "").replace("\r\n", "\n").replace("\r", "\n")

    # Remove emoji/pictographs
    t = re.sub(
        r"[\U0001F300-\U0001F5FF\U0001F600-\U0001F64F"
        r"\U0001F680-\U0001F6FF\U0001F700-\U0001F77F"
        r"\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF"
        r"\U0001F900-\U0001F9FF\U0001FA00-\U0001FAFF"
        r"\U00002700-\U000027BF\U0001F1E6-\U0001F1FF]+",
        "",
        t,
    )

    # Split bullets and numbered lists
    t = re.sub(r"^\s*[-*•]+\s*", f"{SPLIT_TOKEN} ", t)
    t = re.sub(r"\n\s*[-*•]+\s*", f" {SPLIT_TOKEN} ", t)
    t = re.sub(r"(?:(?<=^)|(?<=\s))-(?=[A-Za-z])", f" {SPLIT_TOKEN} ", t)
    t = re.sub(r"\n\s*\d{1,3}[\.)]\s*", f" {SPLIT_TOKEN} ", t)
    t = re.sub(r"(?:^|(?<=\s))\d{1,3}[\.)](?=\s|[A-Za-z])\s*", f" {SPLIT_TOKEN} ", t)

    # Split line breaks and semicolons
    t = re.sub(r"\n+", f" {SPLIT_TOKEN} ", t)
    t = re.sub(r";+\s*", f" {SPLIT_TOKEN} ", t)

    # Protect decimals
    t = re.sub(r"(\d)\.(\d)", rf"\1{DEC_TOKEN}\2", t)
    return t

def protect_abbreviations(text):
    out = text
    for abbr in ABBREVIATIONS:
        escaped = re.escape(abbr)
        repl = abbr.replace(".", DOT_TOKEN)
        out = re.sub(escaped, repl, out, flags=re.IGNORECASE)
    return out

def protect_hyphen_in_parentheses(text):
    return re.sub(
        r"\([^()]*\)",
        lambda m: m.group(0).replace("-", PAREN_HYPHEN_TOKEN),
        text
    )

def protect_pros_cons_dash(text):
    pattern = r"(?i)\b(pros?|cons?)\s*:?\s*(?:-\s*){1,3}"
    return re.sub(pattern, lambda m: f"{m.group(1)} {PROCON_DASH_TOKEN} ", text)

def protect_day_ranges(text):
    pattern = (
        r"(?i)\b("
        r"mon(?:day)?|tue(?:s|sday)?|wed(?:nesday)?|thu(?:r|rs|rsday)?|"
        r"fri(?:day)?|sat(?:urday)?|sun(?:day)?"
        r")\s*-\s*("
        r"mon(?:day)?|tue(?:s|sday)?|wed(?:nesday)?|thu(?:r|rs|rsday)?|"
        r"fri(?:day)?|sat(?:urday)?|sun(?:day)?"
        r")\b"
    )
    return re.sub(
        pattern,
        lambda m: f"{m.group(1)} {DAY_RANGE_HYPHEN_TOKEN} {m.group(2)}",
        text
    )

def protect_label_colon_dash(text):
    pattern = r"\b([A-Za-z][A-Za-z/& ]{1,40})\s*:\s*-\s+"
    return re.sub(
        pattern,
        lambda m: f"{m.group(1).strip()}: {LABEL_COLON_DASH_TOKEN} ",
        text
    )

def split_before_label_colon_dash(text):
    pattern = (
        r"(?i)\s+(?="
        r"(?:what|the|why|how|my|overall|advice|areas|benefit|benefits|pros?|cons?)"
        r"(?:\s+[A-Za-z][A-Za-z/&]*){0,8}"
        r"\s*:\s*<LABEL_COLON_DASH_KEEP>\s*"
        r")"
    )
    return re.sub(pattern, f" {SPLIT_TOKEN} ", text)

def protect_conjunction_comma_dash(text):
    pattern = r"(?i)\b(but|and|so|or|however|therefore),\s*-\s+"
    return re.sub(
        pattern,
        lambda m: f"{m.group(1)}, {CONJ_COMMA_DASH_TOKEN} ",
        text
    )

def split_inline_dash_bullets(text):
    return re.sub(r"\s-\s(?=(?:[A-Za-z]|\())", f" {SPLIT_TOKEN} ", text)

def layer2_split(text):
    t = protect_hyphen_in_parentheses(text)
    t = protect_pros_cons_dash(t)
    t = protect_day_ranges(t)
    t = protect_label_colon_dash(t)
    t = protect_conjunction_comma_dash(t)
    t = split_before_label_colon_dash(t)
    t = split_inline_dash_bullets(t)

    t = re.sub(
        r"(?i)\s+(?=(?:pro|pros|con|cons)\s*(?::|-))",
        f" {SPLIT_TOKEN} ",
        t
    )

    t = protect_abbreviations(t)

    t = re.sub(r"([!?]+)(\s+|$)", rf"\1 {SPLIT_TOKEN} ", t)
    t = re.sub(r"((?<!\.)\.(?!\.))(\s+|$)", rf"\1 {SPLIT_TOKEN} ", t)
    t = re.sub(r"([.!?])([A-Za-z])", rf"\1 {SPLIT_TOKEN} \2", t)

    t = t.replace(DOT_TOKEN, ".")
    t = t.replace(DEC_TOKEN, ".")
    t = t.replace(PAREN_HYPHEN_TOKEN, "-")
    t = t.replace(PROCON_DASH_TOKEN, "-")
    t = t.replace(DAY_RANGE_HYPHEN_TOKEN, "-")
    t = t.replace(LABEL_COLON_DASH_TOKEN, "-")
    t = t.replace(CONJ_COMMA_DASH_TOKEN, "-")

    parts = [clean_space(p) for p in t.split(SPLIT_TOKEN)]
    parts = [p for p in parts if p and p not in {"-", "*", "•"}]

    out = []
    for p in parts:
        if p.count(">") >= 2:
            out.extend([
                clean_space(x)
                for x in re.split(r"\s*>\s*", p)
                if clean_space(x)
            ])
        else:
            out.append(p)

    return out

def split_text_to_sentences(text):
    if pd.isna(text) or not str(text).strip():
        return []

    stage1 = layer1_prepare_and_split(str(text))
    return layer2_split(stage1)

## Create sentence-level dataframe

In [ ]:
sentence_rows = []

for _, row in nlp_df.iterrows():
    sentences = split_text_to_sentences(row["text"])

    for sentence_id, sentence in enumerate(sentences, start=1):
        sentence_rows.append({
            "review_row_id": row["review_row_id"],
            "id": row["id"],
            "company_name": row["company_name"],
            "text_content": row["text_content"],
            "sentence_id": sentence_id,
            "sentence_text": sentence,
            "original_text": row["text"]
        })

sentence_df = pd.DataFrame(sentence_rows)

sentence_df = (
    sentence_df
    .sort_values(["id", "review_row_id", "text_content", "sentence_id"])
    .reset_index(drop=True)
)

print("Original NLP rows:", len(nlp_df))
print("Sentence-level rows:", len(sentence_df))
display(sentence_df.head())

## Remove single-word fragments

In [ ]:
rows_before = len(sentence_df)

sentence_df = sentence_df[
    sentence_df["sentence_text"].astype(str).str.split().str.len() > 1
].reset_index(drop=True)

rows_after = len(sentence_df)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Removed:", rows_before - rows_after)

## Export for manual inspection

Open the exported CSV, inspect the sentence boundaries, correct any incorrect splits, and save the manually checked file as:

`data/intermediate/02_sentence_level_manual_inspection.csv`

In [ ]:
sentence_output = INTERMEDIATE_DIR / "01_sentence_level.csv"
manual_output = INTERMEDIATE_DIR / "02_sentence_level_manual_inspection.csv"

sentence_df.to_csv(sentence_output, index=False, encoding="utf-8-sig")

print("Saved:", sentence_output)
print("After manual inspection, save your checked file as:")
print(manual_output)